<a href="https://colab.research.google.com/github/emmanuelmassawe/breast-cancer-predictions-with-pytorch/blob/main/breast_cancer_predictions_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset,DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
data = load_breast_cancer()
X, y = data.data, data.target

In [4]:
X_train,X_test ,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42 , stratify=y)


In [5]:
print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
X_train = torch.tensor(X_train,dtype=torch.float32).to(device)
X_test = torch.tensor(X_test,dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype = torch.long).to(device)
y_test = torch.tensor(y_test, dtype = torch.long).to(device)



In [8]:
train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)

train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size= 32, shuffle = False)

creating the Neural Network Architecture


In [20]:
class Nn(nn.Module):
  def __init__(self):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(30,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,2)
    )
  def forward(self,x):
    return self.network(x)

model = Nn()
print(model)

Nn(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=2, bias=True)
  )
)


Apply the loss function

In [15]:
loss_fn = nn.BCEWithLogitsLoss()

applying the optimizer

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

train the model

In [21]:
epoch = 100

for i in range(epoch):
  model.train()
  for X_batch,y_batch in train_loader:
    optimizer.zero_grad()
    y_pred = model(X_batch)
    loss = loss_fn(y_pred,y_batch)


    loss.backward()
    optimizer.step()

    if (i+1) % 10 == 0:
      print(f"Epoch {i+1}/{epoch}, Loss: {loss.item():.4f}")



RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)